# Beca 18 RAG Chatbot
**Fuente:** Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC  
**Modelo de embeddings:** `gemini-embedding-001` | **Modelo de generación:** `gemini-2.5-flash`

El sistema responde **únicamente** con información contenida en el documento oficial. Si la información no existe en el documento, responde: *"The document does not contain information about this topic."*

## Step 0 — Setup

In [ ]:
!pip install -q pypdf tiktoken langchain-text-splitters google-genai chromadb ipywidgets tqdm python-dotenv

In [ ]:
import os
import re
import pypdf
import tiktoken
import chromadb
import ipywidgets
import google.genai
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Detect environment and load API key accordingly
try:
    from google.colab import userdata
    IN_COLAB = True
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv
    load_dotenv()
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

assert GEMINI_API_KEY, "GEMINI_API_KEY not found — add it to Colab Secrets or to your local .env file"

print(f"Environment      : {'Google Colab' if IN_COLAB else 'Local'}")
print(f"pypdf            : {pypdf.__version__}")
print(f"tiktoken         : {tiktoken.__version__}")
print(f"chromadb         : {chromadb.__version__}")
print(f"ipywidgets       : {ipywidgets.__version__}")
print(f"google-genai     : {google.genai.__version__}")
print("API key loaded   : OK")

## Step 1 — PDF Text Extraction

In [ ]:
# Set PDF path — clone repo automatically if running in Colab
if IN_COLAB:
    if not os.path.exists("beca18-rag-chatbot"):
        os.system("git clone https://github.com/aaccasanih-wq/beca18-rag-chatbot.git")
    PDF_PATH = "beca18-rag-chatbot/data/beca18_reglamento.pdf"
else:
    PDF_PATH = "../data/beca18_reglamento.pdf"

def extract_text_from_pdf(path: str) -> str:
    reader = pypdf.PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        raw = page.extract_text() or ""
        raw = re.sub(r" {2,}", " ", raw)
        raw = re.sub(r"(?<!\n)\n(?!\n)", " ", raw)
        raw = raw.strip()
        pages.append(f"[PAGE {i}]\n{raw}")
    return "\n\n".join(pages)

full_text = extract_text_from_pdf(PDF_PATH)

char_count = len(full_text)
word_count = len(full_text.split())
print(f"Total characters : {char_count:,}")
print(f"Total words      : {word_count:,}")
print(f"\n--- Preview (first 500 chars) ---\n{full_text[:500]}")

## Step 2 — Tokenization and Chunking

In [ ]:
# Token count
enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(full_text)
print(f"Total tokens (cl100k_base): {len(tokens):,}")

### Justificación del tamaño de chunk

Se eligió **`chunk_size=400` caracteres** con **`chunk_overlap=60`** por las siguientes razones:

- El modelo `gemini-embedding-001` tiene un límite de **8,192 tokens** por input. Con ~4 caracteres/token en promedio, 400 caracteres equivalen a ~100 tokens — muy por debajo del límite y suficiente para capturar una idea completa del reglamento.
- El **overlap de 60 caracteres** (~15 tokens) evita cortar frases con información crítica (montos, condiciones, plazos) que quedarían truncadas entre dos chunks contiguos.
- Chunks más grandes (ej. 800+) reducen la precisión del retrieval al mezclar múltiples temas en un solo vector. Chunks más pequeños (ej. 100) pierden contexto semántico suficiente para responder preguntas de manera coherente.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "],
)

raw_chunks = splitter.split_text(full_text)

# Attach metadata to each chunk
chunks = [
    {
        "text": chunk,
        "metadata": {
            "document": "RDE-033-2026-MINEDU-VMGI-PRONABEC",
            "topic": "Beca 18 y Becas Especiales - Convocatoria 2026",
            "language": "es",
        },
    }
    for chunk in raw_chunks
]

avg_len = sum(len(c["text"]) for c in chunks) / len(chunks)
print(f"Total chunks     : {len(chunks):,}")
print(f"Avg chunk length : {avg_len:.0f} characters")

## Step 3 — Embeddings

In [ ]:
import time

client = google.genai.Client(api_key=GEMINI_API_KEY)
EMBEDDING_MODEL = "gemini-embedding-001"

def embed_with_backoff(texts: list[str], task_type: str, max_retries: int = 6) -> list[list[float]]:
    delay = 5
    for attempt in range(max_retries):
        try:
            response = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=texts,
                config=google.genai.types.EmbedContentConfig(
                    task_type=task_type,
                    output_dimensionality=768,
                ),
            )
            return [e.values for e in response.embeddings]
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            print(f"Rate limit hit, retrying in {delay}s... ({e})")
            time.sleep(delay)
            delay *= 2

def embed_documents(texts: list[str]) -> list[list[float]]:
    return embed_with_backoff(texts, task_type="RETRIEVAL_DOCUMENT")

def embed_query(text: str) -> list[float]:
    return embed_with_backoff([text], task_type="RETRIEVAL_QUERY")[0]

# Smoke test — embed a single chunk and verify dimensions
test_embedding = embed_documents([chunks[0]["text"]])
print(f"Embedding model  : {EMBEDDING_MODEL}")
print(f"Embedding dim    : {len(test_embedding[0])} (expected 768)")

## Step 4 — Vector Database

In [ ]:
from tqdm import tqdm

CHROMA_PATH = "chroma_db_beca18" if IN_COLAB else "../chroma_db_beca18"
COLLECTION_NAME = "beca18"
BATCH_SIZE = 50

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

if collection.count() > 0:
    print(f"Collection already indexed — skipping embedding ({collection.count()} docs loaded)")
else:
    texts    = [c["text"] for c in chunks]
    metas    = [c["metadata"] for c in chunks]
    ids      = [f"chunk_{i}" for i in range(len(chunks))]

    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Indexing"):
        batch_texts = texts[i : i + BATCH_SIZE]
        batch_metas = metas[i : i + BATCH_SIZE]
        batch_ids   = ids[i : i + BATCH_SIZE]
        batch_embeddings = embed_documents(batch_texts)
        collection.add(
            ids=batch_ids,
            documents=batch_texts,
            embeddings=batch_embeddings,
            metadatas=batch_metas,
        )
        time.sleep(15)  # respect free-tier limit of 5 req/min

print(f"Total documents in collection: {collection.count()}")

## Step 5 — Semantic Search

In [ ]:
def semantic_search(question: str, k: int = 5) -> list[dict]:
    query_embedding = embed_query(question)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        }
        for i in range(len(results["documents"][0]))
    ]

# Test
test_question = "¿Cuáles son los requisitos para postular a la Beca 18?"
results = semantic_search(test_question, k=5)

print(f"Query: {test_question}\n")
for i, r in enumerate(results[:3], 1):
    print(f"--- Result {i} (distance: {r['distance']:.4f}) ---")
    print(r["text"][:300])
    print()

## Step 6 — Grounded Generation

In [ ]:
GENERATION_MODEL = "gemini-2.5-flash"

SYSTEM_PROMPT = """Eres un asistente experto en el reglamento de Beca 18 de PRONABEC.
Responde ÚNICAMENTE con información contenida en los fragmentos del documento que se te proporcionan como contexto.
- Cita siempre el número de página entre corchetes, por ejemplo: [PAGE 5].
- Si la información no está en el contexto proporcionado, responde exactamente: "The document does not contain information about this topic."
- No uses conocimiento externo ni hagas suposiciones más allá del contexto."""

def answer_with_context(question: str, k: int = 5) -> dict:
    context_chunks = semantic_search(question, k=k)
    context_text = "\n\n".join(
        f"[Fragmento {i+1}]\n{c['text']}" for i, c in enumerate(context_chunks)
    )
    prompt = f"""{SYSTEM_PROMPT}

CONTEXTO DEL DOCUMENTO:
{context_text}

PREGUNTA: {question}

RESPUESTA:"""

    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt,
    )
    return {
        "question": question,
        "answer": response.text,
        "sources": context_chunks,
    }

# Tests: 5 preguntas sobre el documento + 1 fuera de tema
test_questions = [
    "¿Cuáles son los requisitos de elegibilidad para postular a la Beca 18?",
    "¿Qué modalidades de beca existen en la convocatoria 2026?",
    "¿Cuál es el monto de la subvención económica que otorga la beca?",
    "¿Cuáles son las obligaciones del becario una vez adjudicada la beca?",
    "¿Bajo qué causales se puede perder la Beca 18?",
    "¿Cuál es la receta del ceviche peruano?",  # hallucination test
]

for q in test_questions:
    result = answer_with_context(q)
    print(f"PREGUNTA: {result['question']}")
    print(f"RESPUESTA: {result['answer']}")
    print("-" * 80)

## Step 7 — Interactive Chat UI

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Widgets ---
question_input = widgets.Text(
    placeholder="Escribe tu pregunta sobre Beca 18...",
    layout=widgets.Layout(width="70%"),
)
k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="Fragmentos (k):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="40%"),
)
ask_button   = widgets.Button(description="Preguntar", button_style="primary")
clear_button = widgets.Button(description="Limpiar",   button_style="warning")
output_area  = widgets.Output()

def on_ask(b):
    question = question_input.value.strip()
    if not question:
        return
    with output_area:
        clear_output()
        print("Buscando respuesta...")
        result = answer_with_context(question, k=k_slider.value)

        clear_output()
        print(f"Respuesta:\n{result['answer']}\n")

        items = [
            widgets.HTML(
                f"<b>Fragmento {i+1}</b> | Distancia: {s['distance']:.4f}<br>"
                f"<pre style='white-space:pre-wrap'>{s['text'][:400]}</pre>"
            )
            for i, s in enumerate(result["sources"])
        ]
        accordion = widgets.Accordion(children=[widgets.VBox(items)])
        accordion.set_title(0, "Ver fragmentos fuente")
        display(accordion)

def on_clear(b):
    question_input.value = ""
    with output_area:
        clear_output()

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

display(
    widgets.VBox([
        widgets.HTML("<h3>Chatbot Beca 18 — RAG</h3>"),
        widgets.HBox([question_input, ask_button, clear_button]),
        k_slider,
        output_area,
    ])
)